In [124]:
import pandas as pd
import numpy as np

In [125]:
df = pd.read_csv('/content/course_lead_scoring_2026.csv')

In [126]:
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [127]:
df.columns

Index(['lead_source', 'industry', 'employment_status', 'location',
       'annual_income', 'number_of_courses_viewed', 'interaction_count',
       'lead_score', 'converted'],
      dtype='object')

In [128]:
df.isnull().sum()

,0
lead_source,148
industry,240
employment_status,194
location,208
annual_income,369
number_of_courses_viewed,0
interaction_count,0
lead_score,35
converted,0


In [129]:
df.dtypes

,0
lead_source,object
industry,object
employment_status,object
location,object
annual_income,float64
number_of_courses_viewed,int64
interaction_count,int64
lead_score,float64
converted,int64


In [130]:
categorical_columns = ['lead_source', 'industry', 'employment_status', 'location']
numerical_columns = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']

In [131]:
for c in categorical_columns:
 df[c] = df[c].fillna('NA')

In [132]:
for n in numerical_columns:
  df[n] = df[n].fillna(0.0)

In [133]:
df.isnull().sum()

,0
lead_source,0
industry,0
employment_status,0
location,0
annual_income,0
number_of_courses_viewed,0
interaction_count,0
lead_score,0
converted,0


In [134]:
industry_mode = df['industry'].mode()

In [135]:
print(industry_mode)

0    technology
Name: industry, dtype: object


In [136]:
df[numerical_columns].corr()

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


In [137]:
from sklearn.model_selection import train_test_split

In [138]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

In [139]:
len(df_train), len(df_val), len(df_test)

(3000, 1000, 1000)

In [140]:
df_train.reset_index(drop=True)
df_val.reset_index(drop=True)
df_test.reset_index(drop=True)

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,social_media,retail,employed,south_america,41247.0,3,7,0.61,1
1,organic_search,healthcare,employed,north_america,70825.0,3,6,0.62,0
2,social_media,healthcare,employed,south_america,58488.0,3,4,0.56,1
3,NA,education,NA,europe,30138.0,1,4,0.48,0
4,NA,retail,employed,asia,53869.0,2,3,0.49,1
...,...,...,...,...,...,...,...,...,...
995,organic_search,manufacturing,employed,NA,65099.0,1,4,0.53,1
996,social_media,healthcare,student,europe,38954.0,0,0,0.15,0
997,organic_search,education,self_employed,asia,48768.0,2,5,0.47,1
998,social_media,NA,student,asia,0.0,2,2,0.35,1


In [141]:
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values


In [142]:
del df_train['converted']
del df_val['converted']
del df_test['converted']


In [143]:
from sklearn.metrics import mutual_info_score

In [144]:
def mutual_info_converted(series):
    score = mutual_info_score(series, df_full_train.converted)
    return round(score, 2)

df_full_train[categorical_columns].apply(mutual_info_converted)

,0
lead_source,0.03
industry,0.00
employment_status,0.02
location,0.00


In [145]:
from sklearn.feature_extraction import DictVectorizer
train_dicts = df_train[categorical_columns + numerical_columns].to_dict(orient='records')
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)
val_dicts = df_val[categorical_columns + numerical_columns].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [146]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [148]:
y_pred = model.predict_proba(X_val)[:, 1]
percentage_converted = (y_pred >= 0.5)
score = (y_val == percentage_converted).mean()
score

np.float64(0.645)

In [149]:
feature_names = dv.get_feature_names_out()
feature_names

array(['annual_income', 'employment_status=NA',
       'employment_status=employed', 'employment_status=self_employed',
       'employment_status=student', 'employment_status=unemployed',
       'industry=NA', 'industry=education', 'industry=finance',
       'industry=healthcare', 'industry=manufacturing', 'industry=retail',
       'industry=technology', 'interaction_count', 'lead_score',
       'lead_source=NA', 'lead_source=events',
       'lead_source=organic_search', 'lead_source=paid_ads',
       'lead_source=referral', 'lead_source=social_media', 'location=NA',
       'location=africa', 'location=asia', 'location=europe',
       'location=north_america', 'location=south_america',
       'number_of_courses_viewed'], dtype=object)

In [150]:
features = categorical_columns + numerical_columns

In [154]:
for feature in features:
    features_list = []
    for f in features:
        if f != feature:
            features_list.append(f)

    train_dicts = df_train[features_list].to_dict(orient='records')
    val_dicts = df_val[features_list].to_dict(orient='records')

    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(train_dicts)
    X_val = dv.transform(val_dicts)

    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict_proba(X_val)[:, 1]
    percentage_converted = (y_pred >= 0.5)
    score_1 = (y_val == percentage_converted).mean()
    print("Excluded feature:", feature, "Accuracy:", score_1)

Excluded feature: lead_source Accuracy: 0.642
Excluded feature: industry Accuracy: 0.645
Excluded feature: employment_status Accuracy: 0.645
Excluded feature: location Accuracy: 0.645
Excluded feature: annual_income Accuracy: 0.724
Excluded feature: number_of_courses_viewed Accuracy: 0.643
Excluded feature: interaction_count Accuracy: 0.601
Excluded feature: lead_score Accuracy: 0.644


In [156]:
original_accuracy = score
difference_btw_scores = original_accuracy - score_1
print("Removed feature:", feature)
print("New accuracy:", (score_1))
print("smallest difference:", (difference_btw_scores))
print()

Removed feature: lead_score
New accuracy: 0.644
smallest difference: 0.0



In [159]:
C_values = [0.000001, 0.00001, 0.0001, 0.001]

for C in C_values:

  train_dicts = df_train[categorical_columns + numerical_columns].to_dict(orient='records')
  dv = DictVectorizer(sparse=False)
  X_train = dv.fit_transform(train_dicts)
  val_dicts = df_val[categorical_columns + numerical_columns].to_dict(orient='records')
  X_val = dv.transform(val_dicts)

  model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
  model.fit(X_train, y_train)

  y_pred = model.predict_proba(X_val)[:, 1]
  percentage_converted = (y_pred >= 0.5)
  score = (y_val == percentage_converted).mean()
  print("C:", C, "Accuracy:", round(score, 3))

C: 1e-06 Accuracy: 0.598
C: 1e-05 Accuracy: 0.598
C: 0.0001 Accuracy: 0.613
C: 0.001 Accuracy: 0.645
